# Module 5 — Deployment

The web service itself lives in `predict.py`, not in this notebook: a notebook
cell that calls `uvicorn.run(...)` blocks the kernel forever and can never be
reached by the client cell further down.


## Q1 — install uv

In [5]:
!pipx install uv
!uv --version

fish: Unknown command: pipx
fish: 
pipx install uv
^~~^
uv 0.12.13 (0ebbd9274 2026-09-10 x86_64-unknown-linux-gnu)


In [26]:
!uv init --bare --name lead-scoring --python ">=3.10" --no-workspace

error: Project is already initialized in
       `/mnt/data/projects/PycharmProjects/MLZoomcamp/homework`
       (`pyproject.toml` file exists)


## Q2 — add scikit-learn and read the lock file

Use `uv add`, not `uv pip install`. `uv pip install` drops the package into the
environment without touching `../../pyproject.toml` or `uv.lock`, so the Docker build
later (`uv sync --locked`) would come up without scikit-learn.

In [28]:
!uv add scikit-learn==1.6.1

  × No solution found when resolving dependencies for split (markers:               
  │ python_full_version >= '3.14' and sys_platform == 'win32'):
  ╰─▶ Because homework depends on scikit-learn==1.6.1 and mlzoomcamp depends
      on scikit-learn>=1.9.0, we can conclude that homework and mlzoomcamp
      are incompatible.
      And because your workspace requires homework and mlzoomcamp, we can
      conclude that your workspace's requirements are unsatisfiable.

hint: The resolution failed for an environment that is not the current one, consider limiting the environments with `tool.uv.environments`.
hint: If you want to add the package regardless of the failed resolution, provide the `--frozen` flag to skip locking and syncing


In [29]:
import json

with open("uv.lock", "rb") as f_in:
    lock = f_in.read().decode()

# first hash recorded for the scikit-learn wheel
start = lock.index('name = "scikit-learn"')
print(lock[start:start + 800])

name = "scikit-learn" },
    { name = "uvicorn", extra = ["standard"] },
]

[package.metadata]
requires-dist = [
    { name = "fastapi", specifier = ">=0.141.1" },
    { name = "requests", specifier = ">=2.34.2" },
    { name = "scikit-learn", specifier = "==1.6.1" },
    { name = "uvicorn", extras = ["standard"], specifier = ">=0.52.4" },
]

[[package]]
name = "numpy"
version = "2.2.6"
source = { registry = "https://pypi.org/simple" }
resolution-markers = [
    "python_full_version < '3.11'",
]
sdist = { url = "https://files.pythonhosted.org/packages/76/21/7d2a95e4bba9dc13d043ee156a356c0a8f0c6309dff6b21b4d71a073b8a8/numpy-2.2.6.tar.gz", hash = "sha256:e29554e2bef54a90aa5cc07da6ce955accb83f21ab5de01a62c8478897b264fd", size = 20276440, upload-time = "2025-05-17T22:38:04.611Z" }
wheels = [
 


## Q3 — score one lead

In [30]:
!wget -nc https://github.com/DataTalksClub/machine-learning-zoomcamp/raw/refs/heads/main/cohorts/2025/05-deployment/homework/pipeline_v1.bin

File ‘pipeline_v1.bin’ already there; not retrieving.



In [31]:
import pickle

with open("pipeline_v1.bin", "rb") as f_in:
    pipeline = pickle.load(f_in)

pipeline

/mnt/data/projects/PycharmProjects/MLZoomcamp/.venv/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/mnt/data/projects/PycharmProjects/MLZoomcamp/.venv/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/mnt/data/projects/PycharmProjects/MLZoomcamp/.venv/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dictvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"dtype dtype: dtype, default=np.float64The type of feature values. Passed to Numpy array/scipy.sparse matrixconstructors as the dtype argument.",<class 'numpy.float64'>
,"separator separator: str, default=""=""Separator string used when constructing new features for one-hotcoding.",'='
,"sparse sparse: bool, default=TrueWhether transform should produce scipy.sparse matrices.",True
,"sort sort: bool, default=TrueWhether ``feature_names_`` and ``vocabulary_`` should besorted when fitting.",True
Name,Type,Value


In [32]:
lead = {
    "lead_source": "paid_ads",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
}

# The pipeline starts with a DictVectorizer, which expects an iterable of
# dicts. Passing the bare dict raises inside the vectorizer.
probability = pipeline.predict_proba([lead])[0, 1]
print(round(probability, 3))

0.534


## Q4 — serve it with FastAPI

In [33]:
!uv add fastapi 'uvicorn[standard]' requests

Resolved 164 packages in 0.69ms
Prepared 1 package in 5ms                                                
Uninstalled 1 package in 0.31ms
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 0.93msfile:///mnt/data/projects/Pycha
 ~ homework==0.1.0 (from file:///mnt/data/projects/PycharmProjects/MLZoomcamp/homework)


Write the service out to its own module so it can be imported by uvicorn and copied into the image.

In [34]:
%%writefile predict.py
"""Lead-scoring prediction service.

Run locally:
    uv run uvicorn predict:app --host 0.0.0.0 --port 9696

The model path can be overridden with the MODEL_PATH environment variable,
which is how the Docker image points at the model baked into the base image.
"""

import os
import pickle

from fastapi import FastAPI
from pydantic import BaseModel

MODEL_PATH = os.getenv("MODEL_PATH", "pipeline_v1.bin")

with open(MODEL_PATH, "rb") as f_in:
    pipeline = pickle.load(f_in)

app = FastAPI(title="lead-scoring")


class Lead(BaseModel):
    lead_source: str
    number_of_courses_viewed: int
    annual_income: float


@app.post("/predict")
def predict(lead: Lead):
    # DictVectorizer expects an iterable of dicts, never a single dict.
    features = [lead.model_dump()]
    probability = float(pipeline.predict_proba(features)[0, 1])
    return {"probability": probability, "converted": probability >= 0.5}


@app.get("/health")
def health():
    return {"status": "ok"}


if __name__ == "__main__":
    import uvicorn

    uvicorn.run(app, host="0.0.0.0", port=9696)


Overwriting predict.py


Start the server in a terminal (not in a cell — it never returns):

```bash
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

Or, from the notebook, run it in the background so the client cell below can reach it:

In [35]:
import subprocess, time

server = subprocess.Popen(
    ["uv", "run", "uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]
)
time.sleep(5)

/mnt/data/projects/PycharmProjects/MLZoomcamp/.venv/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/mnt/data/projects/PycharmProjects/MLZoomcamp/.venv/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/mnt/data/projects/PycharmProjects/MLZoomcamp/.venv/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning

In [36]:
import requests

# The URL needs the scheme and the route — 'localhost:9696' alone raises
# requests.exceptions.MissingSchema.
url = "http://localhost:9696/predict"

client = {
    "lead_source": "organic_search",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
}

response = requests.post(url, json=client)
response.raise_for_status()
response.json()

INFO:     127.0.0.1:56054 - "POST /predict HTTP/1.1" 200 OK


{'probability': 0.5340417283801275, 'converted': True}

In [37]:
server.terminate()

INFO:     Shutting down


## Q5 — pull the base image

In [13]:
!docker pull zoomcamp-model:2026

Emulate Docker CLI using podman. Create /etc/containers/nodocker to quiet msg.
Error: short-name "zoomcamp-model:2026" did not resolve to an alias and no containers-registries.conf(5) was found


In [12]:
!docker build -t zoomcamp-model:2026-hw5 .
!docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5

Emulate Docker CLI using podman. Create /etc/containers/nodocker to quiet msg.
STEP 1/9: FROM agrigorev/zoomcamp-model:2025
Error: creating build container: short-name "agrigorev/zoomcamp-model:2025" did not resolve to an alias and no containers-registries.conf(5) was found
Emulate Docker CLI using podman. Create /etc/containers/nodocker to quiet msg.
Error: short-name "zoomcamp-model:2026-hw5" did not resolve to an alias and no containers-registries.conf(5) was found


## Q6 — build and run the image

The original Dockerfile stopped after `uv sync`: no application code, no port,
no entrypoint, so the resulting image had nothing to run.

In [14]:
%%writefile Dockerfile
FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/

ENV PATH="/code/.venv/bin:$PATH" \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /code
COPY pyproject.toml uv.lock .python-version ./
RUN uv sync --locked --no-dev

COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./

EXPOSE 9696
ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]

Overwriting Dockerfile


In [17]:
!podman build -t lead-scoring .

STEP 1/9: FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
STEP 2/9: COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/
--> Using cache c619a15132bbd0c4291069d8682fbe90629d99184765ab039311eb49bfcf4d1e
--> c619a15132bb
STEP 3/9: ENV PATH="/code/.venv/bin:$PATH"     UV_COMPILE_BYTECODE=1     UV_LINK_MODE=copy
--> Using cache ed5830f6844cfd2141b199a6488fa463f11d64ab580b7c727f8f97e8ebb63710
--> ed5830f6844c
STEP 4/9: WORKDIR /code
--> Using cache f03acbdf9c68be57c8484a15284e8fef29ef568fb22852bdd7511e252ec2a7e1
--> f03acbdf9c68
STEP 5/9: COPY pyproject.toml uv.lock .python-version ./
--> 395559939bf7
STEP 6/9: RUN uv sync --locked --no-dev
 Downloaded cpython-3.14.3-linux-x86_64-gnu (download)
Using CPython 3.14.3
Creating virtual environment at: .venv
Resolved 24 packages in 0.36ms
   Building homework @ file:///code
 Downloaded pydantic-core
  × Failed to b

In [18]:
!podman run -d --rm -p 9696:9696 --name lead-scoring lead-scoring

Error: short-name "lead-scoring" did not resolve to an alias and no containers-registries.conf(5) was found


In [19]:
import requests, time

time.sleep(3)
requests.post(
    "http://localhost:9696/predict",
    json={
        "lead_source": "organic_search",
        "number_of_courses_viewed": 4,
        "annual_income": 80304.0,
    },
).json()

ConnectionError: HTTPConnectionPool(host='localhost', port=9696): Max retries exceeded with url: /predict (Caused by NewConnectionError("HTTPConnection(host='localhost', port=9696): Failed to establish a new connection: [Errno 111] Connection refused"))

In [ ]:
!docker stop lead-scoring